In [132]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.datasets

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder

In [133]:
df = pd.read_csv("/content/Bengaluru_House_Data.csv")

In [134]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [135]:
df.shape


(13320, 9)

In [136]:
df.describe()

,bath,balcony,price
count,13247.000000,12711.000000,13320.000000
mean,2.692610,1.584376,112.565627
std,1.341458,0.817263,148.971674
min,1.000000,0.000000,8.000000
25%,2.000000,1.000000,50.000000
50%,2.000000,2.000000,72.000000
75%,3.000000,2.000000,120.000000
max,40.000000,3.000000,3600.000000


In [137]:
df.isnull().sum()

,0
area_type,0
availability,0
location,1
size,16
society,5502
total_sqft,0
bath,73
balcony,609
price,0


In [138]:
df["bath"] = df["bath"].fillna(df["bath"].mean())
df["balcony"] = df["balcony"].fillna(df["balcony"].mean())

df["size"] = df["size"].fillna(df["size"].mode()[0])
df["location"] = df["location"].fillna(df["location"].mode()[0])

In [139]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [140]:
df.dropna(inplace = True)

In [141]:
df.shape

(7818, 9)

In [142]:
from sklearn.preprocessing import LabelEncoder

for col in ["area_type", "availability", "location", "society"]:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

df["bhk"] = df["size"].str.extract(r'(\d+)').astype(float)


In [143]:
df.dtypes

,0
area_type,int64
availability,int64
location,int64
size,object
society,int64
total_sqft,object
bath,float64
balcony,float64
price,float64
bhk,float64


In [145]:
# 1. Remove society column
df.drop("society", axis=1, inplace=True)

# 2. Fill missing values
df["location"] = df["location"].fillna(df["location"].mode()[0])
df["size"] = df["size"].fillna(df["size"].mode()[0])

# 3. Convert size to BHK
df["bhk"] = df["size"].str.extract(r'(\d+)').astype(float)
df.drop("size", axis=1, inplace=True)

# 4. Convert total_sqft to numeric
def convert_sqft(x):
    try:
        if '-' in str(x):
            a, b = x.split('-')
            return (float(a) + float(b)) / 2
        return float(x)
    except:
        return np.nan

df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

# 5. Remove rows where sqft couldn't be converted
df.dropna(subset=["total_sqft"], inplace=True)

# 6. Create price_per_sqft
df["price_per_sqft"] = (df["price"] * 100000) / df["total_sqft"]

# 7. Remove price_per_sqft outliers
df = df[
    (df["price_per_sqft"] > 1000) &
    (df["price_per_sqft"] < 30000)
]

# 8. Group rare locations
location_stats = df["location"].value_counts()

df["location"] = df["location"].apply(
    lambda x: "other" if location_stats[x] <= 10 else x
)

# 9. One-hot encode categorical columns
df = pd.get_dummies(
    df,
    columns=["area_type", "availability", "location"],
    drop_first=True
)

# 10. Remove any remaining nulls
df.dropna(inplace=True)

In [146]:
df.dtypes

,0
total_sqft,float64
bath,float64
balcony,float64
price,float64
bhk,float64
...,...
location_639,bool
location_642,bool
location_643,bool
location_646,bool


In [147]:
df.isnull().sum()
df = df[df["price"] < df["price"].quantile(0.99)]

In [148]:

corelation = df.corr()

In [149]:
plt.figure(figsize=(9,9))
sns.heatmap(corelation, annot = True,cbar = True,square = True,fmt = ".1f")
plt.show()

KeyboardInterrupt: 

Error in callback <function flush_figures at 0x7f69cd439bc0> (for post_execute):


KeyboardInterrupt: 

In [150]:
X = df.drop(["price"], axis = 1)
Y = df["price"]

In [151]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

In [152]:
print(X_train.shape,X_test.shape,X.shape)

(6170, 231) (1543, 231) (7713, 231)


In [153]:
model  = XGBRegressor()

In [154]:
model.fit(X_train,Y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [155]:
training_data_predict = model.predict(X_train)

In [156]:

print(training_data_predict)

[62.07391  98.90102  69.9612   ... 69.10721  85.56659  62.882973]


In [157]:
print(Y_train)

4907      62.00
2934     100.00
11799     71.81
4158      75.00
7031     136.00
          ...  
6260      60.00
9873      75.00
11527     68.48
4502      84.95
12702     62.60
Name: price, Length: 6170, dtype: float64


In [158]:
#erorrs

In [159]:
score1 = metrics.r2_score(Y_train, training_data_predict)

In [160]:
score2 = metrics.mean_absolute_error(Y_train, training_data_predict)

In [161]:
print("R square error", score1)
print("Mean absolute error", score2  )

R square error 0.9997078828856234
Mean absolute error 0.8501068045494045


In [162]:
df["price"].mean()

np.float64(93.29218138208219)

In [163]:
#predicton on test data

In [164]:
test_data_predict = model.predict(X_test)

In [165]:
final1 = metrics.r2_score(Y_test, test_data_predict)

In [166]:
final2 = metrics.mean_absolute_error(Y_test, test_data_predict)

In [167]:
print("R square error", final1)
print("Mean absolute error", final2  )

R square error 0.9921199949946566
Mean absolute error 1.99021334217341


In [168]:
print(Y_test)

10345     52.00
10528     68.00
3197     160.00
11176    150.00
5336     124.00
          ...  
12549     71.95
421      103.00
1909      98.00
4205      82.50
9043      35.00
Name: price, Length: 1543, dtype: float64
